# Fine-tuning de Qwen 2.5 3B para clasificación de relevancia RAG

## Objetivo del notebook

Este notebook implementa el **fine-tuning con QLoRA** del modelo Qwen 2.5 3B-Instruct para la tarea de
**clasificación de relevancia de documentos** en el pipeline RAG de NormaBot.

El modelo aprenderá a decidir, dado un par `(consulta, documento)`, si el documento contiene
información útil para responder la consulta. La salida es binaria:
- **`relevante`** → el documento debe incluirse en el contexto de generación
- **`no relevante`** → el documento debe descartarse

## Contexto en NormaBot

El pipeline RAG (`src/rag/main.py`) sigue el flujo:

```
Retrieve (ChromaDB) → Grade (Qwen 2.5 3B via Ollama) → Generate (Bedrock Nova Lite)
```

La etapa de **grading** actual usa el modelo base con prompting directo. Este fine-tuning busca
especializarlo en el dominio legal (EU AI Act, BOE) para reducir falsos positivos en el contexto
de generación. El experimento:

1. Establece un **baseline** con el modelo base + prompting
2. **Fine-tunea** el mismo modelo con QLoRA sobre el dataset de relevancia
3. **Compara** ambos enfoques: Accuracy, Precision, Recall, F1-macro
4. Proporciona el **código de integración** listo para `src/rag/main.py`

# Instalación de librerías y comprobación de entorno


In [ ]:
# Dependencias necesarias — instalar antes de ejecutar este notebook:
#
#   GPU NVIDIA (CUDA 12.4 — RTX 4070 Super / driver >= 525):
#     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
#   Resto del stack:
#     pip install -r requirements/finetuning.txt

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("\nSin GPU: el entrenamiento será muy lento. Se recomienda GPU con >= 8 GB VRAM.")

In [ ]:
import os
import json
import torch
import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
from pathlib import Path
import sys

# Asegura que el directorio de trabajo sea la raíz del repositorio
# (necesario si Jupyter se abrió desde src/finetuning/ en vez de desde la raíz)
_here = Path.cwd()
if not (_here / "requirements").exists():
    for _parent in _here.parents:
        if (_parent / "requirements").exists():
            os.chdir(_parent)
            break
    else:
        raise RuntimeError(
            f"No se encontró la raíz del repositorio desde {_here}.\n"
            "Ejecuta Jupyter desde la raíz del proyecto: jupyter lab"
        )

# Importar funciones auxiliares del módulo local
sys.path.insert(0, str(Path.cwd() / "src" / "finetuning"))
from functions import (  # noqa: E402
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    format_training_prompt, examples_to_hf_dataset,
    load_tokenizer,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

# Dataset de relevancia

El dataset está generado en `data/processed/grading_dataset.jsonl` mediante
`data/generate_grading_dataset.py`. Cada línea es un ejemplo con tres campos:

```json
{"query": "¿Qué sistemas de IA están prohibidos?", "document": "Artículo 5...", "label": "relevante"}
{"query": "¿Qué sistemas de IA están prohibidos?", "document": "Real Decreto 463/2020...", "label": "no relevante"}
```

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `query` | str | Consulta del usuario sobre normativa de IA |
| `document` | str | Fragmento de texto recuperado de ChromaDB |
| `label` | str | `"relevante"` o `"no relevante"` |

**Composición:** 117 queries · 634 ejemplos · 44.6% relevantes / 55.4% no relevantes  
**Fuentes:** EU AI Act (Arts. 5, 6, 9–17, 25–26, 43, 47–53, 72–73, 99, 113), AESIA, RGPD/LOPDGDD  
**Negativos:** hard negatives del dominio (otro artículo legal) + easy negatives (otra ley ajena al dominio)


In [3]:
from pathlib import Path

# El CWD ya fue fijado a la raíz del repositorio en la celda anterior (env-check-05)

# Rutas locales (relativas a la raíz del repositorio)
DATASET_PATH = Path("data/processed/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR   = "src/finetuning/output/qwen-grader-lora"
ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"
MERGED_PATH  = "src/finetuning/output/qwen-grader-merged"

MAX_SEQ_LEN = 512

print(f"Directorio de trabajo: {Path.cwd()}")
print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

Directorio de trabajo: c:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final
Configuración:
  Modelo base:    Qwen/Qwen2.5-3B-Instruct
  Dataset:        data\processed\grading_dataset.jsonl  (existe: True)
  Output dir:     src/finetuning/output/qwen-grader-lora
  Adapter path:   src/finetuning/output/qwen-grader-lora/adapter_final
  Max seq len:    512


In [4]:
all_data = load_grading_dataset(DATASET_PATH)

Dataset cargado: C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\data\processed\grading_dataset.jsonl
  Total ejemplos:  634
  Relevantes:      283 (44.6%)
  No relevantes:   351 (55.4%)

Muestra:
{
  "query": "¿Cuándo es obligatorio recurrir a un organismo notificado para evaluar conformidad de IA?",
  "document": "Artículo 1258 del Código Civil — Contratos. Los contratos se perfeccionan por el mero consentimiento, y desde entonces obligan no sólo al cumplimiento de lo expresamente pactado, sino también a todas las consecuencias que, según su naturaleza, sean conformes a la buena fe, al uso y a la ley.",
  "label": "no relevante"
}


In [5]:
train_data, val_data, test_data = split_dataset(all_data)

Split del dataset:
  Train: 443 ejemplos
  Val:   95 ejemplos
  Test:  96 ejemplos


In [6]:
show_split_stats(train_data, "TRAIN")
show_split_stats(val_data,   "VALIDATION")
show_split_stats(test_data,  "TEST")

print(f"\n{'='*55}")
print("Ejemplo del dataset:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


  TRAIN (443 ejemplos)
label
no relevante    245
relevante       198
  Longitud media query:    95 chars
  Longitud media document: 417 chars

  VALIDATION (95 ejemplos)
label
no relevante    53
relevante       42
  Longitud media query:    96 chars
  Longitud media document: 417 chars

  TEST (96 ejemplos)
label
no relevante    53
relevante       43
  Longitud media query:    92 chars
  Longitud media document: 421 chars

Ejemplo del dataset:
{
  "query": "¿El EU AI Act tiene su propio régimen de sanciones o depende del RGPD?",
  "document": "Artículo 83 RGPD — Condiciones generales para la imposición de multas administrativas. Las infracciones de los principios básicos del tratamiento y los derechos de los interesados se sancionarán con multas de hasta 20.000.000 EUR o el 4 % del volumen de negocio anual global. Otras infracciones (obligaciones del responsable, del encargado, organismos de certificación) se sancionarán con multas de hasta 10.000.000 EUR o el 2 % del volumen de negoc

# Formato del prompt de entrenamiento

Para el fine-tuning usamos el **chat template nativo de Qwen 2.5** (formato `<|im_start|>`).
Cada ejemplo de entrenamiento tiene la forma:

```
<|im_start|>system
Eres un asistente especializado en normativa de IA...<|im_end|>
<|im_start|>user
Consulta: {query}

Documento: {document}

¿Es este documento relevante para responder la consulta?<|im_end|>
<|im_start|>assistant
{label}<|im_end|>
```

El modelo aprende a generar directamente `relevante` o `no relevante` como turno de assistant.


In [7]:
# Carga minima del tokenizador para visualizar el formato del prompt
tokenizer = load_tokenizer(MODEL_ID)

sample_formatted = format_training_prompt(train_data[0], tokenizer)
print("Ejemplo de prompt de entrenamiento:")
print("-" * 65)
print(sample_formatted)
print("-" * 65)
print(f"Longitud: {len(sample_formatted)} chars")

c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando tokenizador...


c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rammu\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizador listo. Vocabulario: 151,643 tokens
Ejemplo de prompt de entrenamiento:
-----------------------------------------------------------------
<|im_start|>system
Eres un asistente especializado en normativa de inteligencia artificial. Tu tarea es determinar si un documento contiene información útil para responder una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). Responde únicamente con 'relevante' o 'no relevante', sin explicación adicional.<|im_end|>
<|im_start|>user
Consulta: ¿El EU AI Act tiene su propio régimen de sanciones o depende del RGPD?

Documento: Artículo 83 RGPD — Condiciones generales para la imposición de multas administrativas. Las infracciones de los principios básicos del tratamiento y los derechos de los interesados se sancionarán con multas de hasta 20.000.000 EUR o el 4 % del volumen de negocio anual global. Otras infracciones (obligaciones del responsable, del encargado, organismos de certificación) se sancionarán con multas de hasta

In [8]:
train_dataset = examples_to_hf_dataset(train_data, tokenizer)
val_dataset   = examples_to_hf_dataset(val_data,   tokenizer)

print(f"Dataset de entrenamiento: {len(train_dataset)} ejemplos")
print(f"Dataset de validacion:    {len(val_dataset)} ejemplos")
print()
print("Primeros 300 chars del primer ejemplo de train:")
print(train_dataset[0]["text"][:300])

Dataset de entrenamiento: 443 ejemplos
Dataset de validacion:    95 ejemplos

Primeros 300 chars del primer ejemplo de train:
<|im_start|>system
Eres un asistente especializado en normativa de inteligencia artificial. Tu tarea es determinar si un documento contiene información útil para responder una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). Responde únicamente con 'relevante' o 'no relevante', 
